In [3]:
%pip install -U langchain langchain-community langchain-text-splitters langchain-openai langchain-chroma pypdf langchain-huggingface sentence-transformers

   ---------------------------------------- 0.0/574.7 kB ? eta -:--:--
   ------------------------------------ --- 524.3/574.7 kB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 574.7/574.7 kB 2.5 MB/s  0:00:00
   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ------------- -------------------------- 262.1/760.7 kB ? eta -:--:--
   ---------------------------------------- 760.7/760.7 kB 2.1 MB/s  0:00:00

  Attempting uninstall: pypdf

    Found existing installation: pypdf 6.21.1

    Uninstalling pypdf-6.21.1:

      Successfully uninstalled pypdf-6.21.1

   ---------------------------------------- 0/4 [pypdf]
   ---------------------------------------- 0/4 [pypdf]
   ---------------------------------------- 0/4 [pypdf]
   ---------------------------------------- 0/4 [pypdf]
   ---------------------------------------- 0/4 [pypdf]
   ---------------------------------------- 0/4 [pypdf]
   ---------------------------------------- 0/4 [pypdf]
   

In [4]:
from dotenv import load_dotenv
load_dotenv()


True

# 1. DATA INGESTION

### Step 1 - PDF DOCUMENT LOADER - PyMUPDFLoader

In [5]:
# Loading Documents
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 

pdf_path = PROJECT_ROOT / "papers" / "FirstPaper.pdf"
print(pdf_path, pdf_path.exists())

loader = PyMuPDFLoader(str(pdf_path))

pdfpages = loader.load()

# Checking no. of pages
print("Total no. of loaded pages in PDF: ", len(pdfpages))
# Content of page-2
print("\nThe content of 2nd page : \n", pdfpages[1].page_content[:200])
# Metadata of page-2
print("\nMetadata of 2nd page : \n", pdfpages[1].metadata)

for i, page in enumerate(pdfpages):
    print(f"\nDocument {i}")
    # print("Page metadata:", page.metadata)
    print("Characters extracted:", len(page.page_content))


c:\Users\Megha Singh\Documents\Python\Projects\fault-diagnosis-research-rag\papers\FirstPaper.pdf True
Total no. of loaded pages in PDF:  9

The content of 2nd page : 
 SINGH AND SHAIK: INCIPIENT FAULT DETECTION IN STATOR WINDINGS OF AN IM USING ST AND SVM
9497
the analysis of current signals for stator fault diagnosis [12].
The application of discrete and stationary

Metadata of 2nd page : 
 {'producer': 'Aspose.Pdf for .NET 8.3.0; modified using iText® 5.5.6 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'Aspose Ltd.', 'creationdate': '2020-10-31T21:01:30+05:30', 'source': 'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\fault-diagnosis-research-rag\\papers\\FirstPaper.pdf', 'file_path': 'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\fault-diagnosis-research-rag\\papers\\FirstPaper.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': 'Incipient Fault Detection in Stator Windings of an Induction Motor Using Stockwell Transform and SVM', 'author': '', 'subject': 'IE

# Observation
```python
1. It has read column-wise from first line of document till last
2. It is not able to read authors name
3. Title read correctly
4. It is not able to read tables
```

### Step 2 - DOCUMENT CHUNKING - RecursiveCharacterTextSplitter

### Cleaning the document
Removing extra characters or white-spaces

In [6]:
import re
from langchain_core.documents import Document

print("Example before clean-up: ", pdfpages[4].page_content[:500])

def clean_up(pdfpages):
    page = pdfpages.page_content
    page = page.replace("\u00ad", "") # removes soft hyphens, which can appear as invisible or unusual line-break artifacts.
    page = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", page) # joins a word split like induc-\ntion into induction
    page = re.sub(r"[ \t]+", " ", page) # changes repeated spaces and tabs into a single space.
    page = re.sub(r"\n{3,}", "\n\n", page) # educes three or more consecutive line breaks to two, preserving paragraph gaps.
    page = page.strip() # removes whitespace from the beginning and end.

    return Document(
            page_content=page,
            metadata=pdfpages.metadata.copy(),
        )

clean_pages = [clean_up(page) for page in pdfpages]
print("Example after clean-up: ", pdfpages[4].page_content[:500])

print("\n Pages cleaned:", len(clean_pages))

Example before clean-up:  9500
IEEE TRANSACTIONS ON INSTRUMENTATION AND MEASUREMENT, VOL. 69, NO. 12, DECEMBER 2020
A. Fault Detection
For the detection of winding faults, the SD of each phase
for the selected band of frequencies is determined. As stated
in Section III, the chosen fault frequencies are 125, 175,
and 225 Hz. A subband of frequencies with ±10 Hz around
the chosen frequencies is selected to encompass the minor
ﬂuctuations. Thus, the subband frequency ranges are Fr1
= [115−135] Hz for the fault frequency 125
Example after clean-up:  9500
IEEE TRANSACTIONS ON INSTRUMENTATION AND MEASUREMENT, VOL. 69, NO. 12, DECEMBER 2020
A. Fault Detection
For the detection of winding faults, the SD of each phase
for the selected band of frequencies is determined. As stated
in Section III, the chosen fault frequencies are 125, 175,
and 225 Hz. A subband of frequencies with ±10 Hz around
the chosen frequencies is selected to encompass the minor
ﬂuctuations. Thus, the subband frequency range

# Observation
```python
1. This stage is light text cleanup. Cleaning will help to make text a whole text without break ups
2. import re loads Python’s regular-expression tools for finding text patterns.
3. won’t repair missing tables or scrambled columns.
4. The function below:
    - Removes soft hyphen characters.
    - Joins words split by a hyphen at a line break.
    - Replaces runs of spaces or tabs with one space.
    - Reduces three or more line breaks to two.
    - Trims whitespace from the start and end.
```


### Chunking Document - RecursiveCharacterTextSplitter

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = splitter.split_documents(clean_pages)

print("Pages:", len(clean_pages))
print("Chunks:", len(chunks))
# print("First chunk metadata:", chunks[0].metadata)
print(chunks[2].page_content)

Pages: 9
Chunks: 51
and stops, high dv/dt voltage surges, mechanical stresses,
thermal overload, and contamination. Insulation failure has a
deteriorating and rapidly propagating tendency. It gradually
starts as turn to turn shorts, causing a large short circuit
current to ﬂow in the windings leading to localized thermal
overloading. Eventually, under continuous motor operation, it
leads to shorting in the same or adjacent coils, phase to phase
or ground faults. Thus, turn-to-turn shorts are a major source
of deterioration of stator windings that culminates into wide
range of faults such as 1) coil-to-coil fault, (2) phase-to-phase
fault, and (3) coil-to-ground fault that may turn severe leading
to catastrophic failure of the motor.
The long persistence of local shorting of turns may lead
to severe faults that can burn off the motor winding in no
Manuscript received January 7, 2020; revised May 25, 2020; accepted
May 30, 2020. Date of publication June 15, 2020; date of current version


In [8]:
chunks

[Document(metadata={'producer': 'Aspose.Pdf for .NET 8.3.0; modified using iText® 5.5.6 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'Aspose Ltd.', 'creationdate': '2020-10-31T21:01:30+05:30', 'source': 'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\fault-diagnosis-research-rag\\papers\\FirstPaper.pdf', 'file_path': 'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\fault-diagnosis-research-rag\\papers\\FirstPaper.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': 'Incipient Fault Detection in Stator Windings of an Induction Motor Using Stockwell Transform and SVM', 'author': '', 'subject': 'IEEE Transactions on Instrumentation and Measurement;2020;69;12;10.1109/TIM.2020.3002444', 'keywords': '', 'moddate': '2021-04-18T03:40:25-04:00', 'trapped': '', 'modDate': "D:20210418034025-04'00'", 'creationDate': "D:20201031210130+05'30'", 'page': 0, 'start_index': 0}, page_content='9496\nIEEE TRANSACTIONS ON INSTRUMENTATION AND MEASUREMENT, VOL. 69, NO. 12, DECEMBER 2020\n

# Observation
```python
1. Every chunk has its own page_content and meta data. This will help to track down particular chunk using metadata
2. This way of chunking, brings headers and footers also in chunks, they waste up space as well as clutter chunks. Need to improve on strategy.
3. Basically, chunks will be a a list of documents, where each document is a chunk of original Doc.
```

### Embedding the Chunks

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                        encode_kwargs={"normalize_embeddings": True}
                                        )

# checking how embeddings would look like
v = embedding_model.embed_query("Stockwell transform for stator fault detection")
print(len(v))
print(v[:5])


c:\Users\Megha Singh\Documents\Python\Projects\fault-diagnosis-research-rag\.venv-faultlit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4289.85it/s]


384
[-0.10646888613700867, 0.010077684186398983, -0.02220597304403782, 0.02350509725511074, 0.060997381806373596]


### Checking the authenticity of embeddings

In [15]:
import numpy as np

sentences = [
    "Stockwell transform is applied to stator current for inter-turn fault detection",  # A
    "Time-frequency analysis of motor current reveals stator winding faults",            # B: same meaning, different words
    "The conference dinner was held at a hotel in Toronto",    
    "S- transform is applied to stator current for inter-turn fault detection"                          # C: unrelated
]

vecs = np.array(embedding_model.embed_documents(sentences))  
# print(vecs[:2,:10])
print("Showing if vectors are normalized:", np.linalg.norm(vecs, axis=1))  # making sure they are of unit length, as normalization=True is already kept.
sim = vecs @ vecs.T  # dot product of unit vectors = cosine similarity
print("This shows the cosine similarity between all vectors :\n", np.round(sim, 3))

# we can also see similarity by using cosine similarity
vector_a = np.array(vecs[0])
vector_b = np.array(vecs[1])

cosine_similarity = np.dot(vector_a, vector_b) / (
    np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
)

print("The cosine similarity bw these 1st and 2nd vectors :", cosine_similarity)

Showing if vectors are normalized: [1.00000005 1.00000001 1.00000003 1.00000005]
This shows the cosine similarity between all vectors :
 [[ 1.     0.566 -0.129  0.841]
 [ 0.566  1.    -0.155  0.551]
 [-0.129 -0.155  1.    -0.106]
 [ 0.841  0.551 -0.106  1.   ]]
The cosine similarity bw these 1st and 2nd vectors : 0.5655036454742268


# Observation
```python
1. Embedding model, in this case, HuggingFaceEmbeddingModel will embed the sentences or chunks with 384 dimension vector.
2. We can find the similarity between any two vectors using Cosine Similarity
       2.1 Cosine similarity measures how close the directions of two embedding vectors are in vector space
       2.2 similar directions suggest the texts have similar meanings or contexts—not necessarily the same words.
3. The vectors having high similarity means they are close in the directions and also have similar meanings.
```

## Storing embeddings in Vector Store

In [11]:
from langchain_chroma import Chroma

vector_store = Chroma(
                collection_name="ResearchLit_Copilot",
                embedding_function= embedding_model
)

try:
    vector_store.reset_collection()
except NameError:
    pass

chunk_ids = vector_store.add_documents(chunks)

print("Chunks supplied:", len(chunks))
print("Chunks indexed:", len(chunk_ids))
print("First chunk ID:", chunk_ids[0])
print(vector_store._collection.count(), len(chunks)) # these should match, if former has more entries means duplicate entries

Chunks supplied: 51
Chunks indexed: 51
First chunk ID: 29ec26a7-cd2e-4af8-872e-e4fe16f2b98f
51 51


# Observation
```python
1. Chroma is a lightweight store, can be easily downloaded and used on laptop or for smaller projects.
2. we have to give the embedding model, collection name to make an object of the class and then use add documents to add chunks to Chroma.
3. We have to reset collection every time, we are running the cell or adding same docs to store, otherwise it will create duplicate entries.
4. It provides chunk ids as the output.
```

In [16]:
chunk_ids

['29ec26a7-cd2e-4af8-872e-e4fe16f2b98f',
 '8533791c-9d76-4cd7-902e-7a9113ae4b5b',
 'e6726c9a-1a15-4f49-9bba-9fd1479dfba3',
 '3777d95c-1846-4e10-81b4-7729f9b24c0a',
 '2442e6a7-8c2c-4819-973a-2aee2fa909f0',
 '90903d76-26b0-4aba-b08c-86247d6f5549',
 'ea35f2bd-85a1-4ac2-9332-6214385e6bcc',
 '952979db-2686-4a0b-8fb5-5ac3d3f6d220',
 '48136c9b-32df-4fe2-9753-320ac4d71b04',
 'b2f9c85d-e039-448b-ab61-99326db47290',
 '9a186b59-3403-4025-96a7-02eba06c0ef8',
 '3daf0155-1e04-45e7-a919-362907243704',
 'aaa368aa-2f0d-4d26-bfe5-7d755c122a84',
 '3d293698-e090-46bd-a61d-3a1e15aae0ca',
 '39824ea2-7a16-43eb-b7db-bc6b8b074846',
 'ee33615d-5748-45b0-a806-c13ce9d4f574',
 '7d160a9b-9646-4d46-875c-35c415e22ec8',
 '9a60b4db-6ebf-4088-a702-d4513abc8c95',
 'a2c34764-a882-4b38-91ff-0912223fca30',
 '0449c27b-dc3e-4a23-b825-82ab5d8a44db',
 '00a532ab-7063-48b3-a8c1-e9beaf863d09',
 '2b47725c-687f-4eba-be13-f9691858a64c',
 '8b9bac39-2ebd-417a-8649-a8c6b7713297',
 '548da56e-2faa-477f-91de-97e1bc79b952',
 '86b1b85b-3b58-

## Retrieval

In [24]:
question = "How does the paper detect bearing faults?"
question = "Which time-frequency method is proposed for stator fault detection?"
retrieved_docs = vector_store.similarity_search_with_score(question, k=5)

for rank, (doc, dist) in enumerate(retrieved_docs, start=1):
    page = doc.metadata["page"] + 1                         # stored 0-indexed -> human page number
    preview = doc.page_content[:100].replace("\n", " ")
    print(f"{rank}. page {page}  dist={dist:.3f}  {preview}...")

for i in (0, 3):
    print(f"--- rank {i+1}, page {retrieved_docs[i][0].metadata['page']+1} ---")
    print(retrieved_docs[i][0].page_content.replace("\n", " "))

1. page 4  dist=0.482  frequencies in (1) have low magnitudes peaks under fault cases only (highlighted). The highlighted f...
2. page 2  dist=0.586  Some articles reported techniques for interturn fault detection, while some other focused on ground ...
3. page 6  dist=0.662  classiﬁcation of the faults which has been detailed in the following stages below. A. Stage 1 Fig. 8...
4. page 4  dist=0.667  each 1-D time-varying signal into time-varying frequency components obtained as a 2-D matrix, where ...
5. page 2  dist=0.677  variation around fault-related frequencies. Two stator winding faults, interturn and ground faults, ...
--- rank 1, page 4 ---
frequencies in (1) have low magnitudes peaks under fault cases only (highlighted). The highlighted frequencies appear mostly in fault cases specially in turn-faults. The time variation of these frequencies can be useful for fault diagnostic purposes. Thus, a wider frequency range can be chosen. In ground faults, there are more peak other th

In [29]:
retrieved_docs

[(Document(id='2b47725c-687f-4eba-be13-f9691858a64c', metadata={'producer': 'Aspose.Pdf for .NET 8.3.0; modified using iText® 5.5.6 ©2000-2015 iText Group NV (AGPL-version)', 'creator': 'Aspose Ltd.', 'format': 'PDF 1.4', 'page': 3, 'start_index': 850, 'author': '', 'file_path': 'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\fault-diagnosis-research-rag\\papers\\FirstPaper.pdf', 'creationdate': '2020-10-31T21:01:30+05:30', 'title': 'Incipient Fault Detection in Stator Windings of an Induction Motor Using Stockwell Transform and SVM', 'source': 'c:\\Users\\Megha Singh\\Documents\\Python\\Projects\\fault-diagnosis-research-rag\\papers\\FirstPaper.pdf', 'keywords': '', 'moddate': '2021-04-18T03:40:25-04:00', 'trapped': '', 'total_pages': 9, 'creationDate': "D:20201031210130+05'30'", 'modDate': "D:20210418034025-04'00'", 'subject': 'IEEE Transactions on Instrumentation and Measurement;2020;69;12;10.1109/TIM.2020.3002444'}, page_content='frequencies in (1) have low magnitudes peaks u

In [30]:
# As the abstract should be the first chunk coming for that question, lets check the rank of this and see whats happening
all_hits = vector_store.similarity_search_with_score(question, k=vector_store._collection.count())
for rank, (doc, dist) in enumerate(all_hits, start=1):
    if doc.metadata["page"] + 1 in (1, 2):
        print(rank, doc.metadata["page"] + 1, f"{dist:.3f}", doc.page_content[:80].replace("\n", " "))

2 2 0.586 Some articles reported techniques for interturn fault detection, while some othe
5 2 0.677 variation around fault-related frequencies. Two stator winding faults, interturn
8 2 0.770 These components are less affected by load variations and not affected by supply
9 1 0.779 have been reported in [11] to be useful for stator interturn shorts detection. N
10 1 0.794 9496 IEEE TRANSACTIONS ON INSTRUMENTATION AND MEASUREMENT, VOL. 69, NO. 12, DECE
11 2 0.798 SINGH AND SHAIK: INCIPIENT FAULT DETECTION IN STATOR WINDINGS OF AN IM USING ST 
13 1 0.816 in [7] compensating the effects of voltage unbalance, inherent asymmetry to dete
15 1 0.884 Manuscript received January 7, 2020; revised May 25, 2020; accepted May 30, 2020
21 2 0.909 components are calculated and given in Table I. Due to voltage unbalances or oth
22 2 0.915 using the Metropolis–Hastings algorithm. Devi et al. [13] and Ali et al. [25] em
33 1 1.032 both types of faults. Under both cases, a heuristic feature selection app

In [31]:
vector_store._collection.count()

51

In [36]:
from collections import Counter
print(Counter(c.metadata["page"] + 1 for c in chunks)) 

page1 = [c for c in chunks if c.metadata["page"] == 0]
for i, c in enumerate(page1):
    print(f"--- page-1 chunk {i} ({len(c.page_content)} chars) ---")
    print(c.page_content[:150].replace("\n", " "), "...\n")

Counter({1: 8, 2: 8, 8: 8, 5: 7, 3: 4, 4: 4, 6: 4, 7: 4, 9: 4})
--- page-1 chunk 0 (1000 chars) ---
9496 IEEE TRANSACTIONS ON INSTRUMENTATION AND MEASUREMENT, VOL. 69, NO. 12, DECEMBER 2020 Incipient Fault Detection in Stator Windings of an Induction ...

--- page-1 chunk 1 (967 chars) ---
both types of faults. Under both cases, a heuristic feature selection approach is utilized to ﬁnd the optimal features for classiﬁcation purposes. Ave ...

--- page-1 chunk 2 (979 chars) ---
and stops, high dv/dt voltage surges, mechanical stresses, thermal overload, and contamination. Insulation failure has a deteriorating and rapidly pro ...

--- page-1 chunk 3 (979 chars) ---
Manuscript received January 7, 2020; revised May 25, 2020; accepted May 30, 2020. Date of publication June 15, 2020; date of current version November  ...

--- page-1 chunk 4 (982 chars) ---
motors, fault detection methods are still in the process of ﬁnalization [5]. Nevertheless, in general, there is some lead time between t

In [ ]:
[c.metadata["page"] + 1 for c in chunks]

# Observation
```python
1. Retrieved docs is a list of documents, each document is a tuple.
2. rank-1 actually has the answer i.e ST, the retriever is correctly working.
```